In [ ]:

# 1) Install Gradio and connect Google Drive
%pip -q install gradio

from google.colab import drive
drive.mount("/content/drive")

# 2) Imports
import os
import json
import numpy as np
import tensorflow as tf
import gradio as gr

from tensorflow import keras
from tensorflow.keras.preprocessing.text import tokenizer_from_json
from PIL import Image

# 3) Model files
MODEL_DIR = "/content/drive/Shareddrives/nti/model1"

encoder_path = os.path.join(MODEL_DIR, "encoder.keras")
decoder_path = os.path.join(MODEL_DIR, "decoder.keras")
tokenizer_path = os.path.join(MODEL_DIR, "tokenizer.json")
config_path = os.path.join(MODEL_DIR, "text_preprocessing_config.json")

for path in [encoder_path, decoder_path, tokenizer_path, config_path]:
    if not os.path.isfile(path):
        raise FileNotFoundError(f"File not found: {path}")

# 4) Load saved model and preprocessing
with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

with open(tokenizer_path, "r", encoding="utf-8") as f:
    tokenizer = tokenizer_from_json(f.read())

MAX_LENGTH = int(config["max_length"])
START_TOKEN = config.get("start_token", "<start>")
END_TOKEN = config.get("end_token", "<end>")
UNK_TOKEN = config.get("unknown_token", "<unk>")

start_id = tokenizer.word_index.get(START_TOKEN)
end_id = tokenizer.word_index.get(END_TOKEN)
unk_id = tokenizer.word_index.get(UNK_TOKEN, 1)

if start_id is None or end_id is None:
    raise ValueError(
        f"Special tokens not found. start_id={start_id}, end_id={end_id}. "
        "Check tokenizer.json and preprocessing config."
    )

encoder_model = keras.models.load_model(
    encoder_path, compile=False, safe_mode=False
)
decoder_model = keras.models.load_model(
    decoder_path, compile=False, safe_mode=False
)

print("Encoder and decoder loaded successfully.")
print("Max sequence length:", MAX_LENGTH)

# 5) Image preprocessing (same 224x224 grayscale input)
def preprocess_uploaded_image(image):
    image = image.convert("L")
    image = image.resize((224, 224), Image.Resampling.BILINEAR)
    arr = np.asarray(image, dtype=np.float32) / 255.0
    return arr[..., np.newaxis][np.newaxis, ...]

# 6) Generate report
def generate_report(frontal_image, lateral_image):
    if frontal_image is None and lateral_image is None:
        raise gr.Error("Please upload at least one X-ray image.")

    # If one image is missing, use the uploaded image for both views.
    if frontal_image is None:
        frontal_image = lateral_image
    if lateral_image is None:
        lateral_image = frontal_image

    frontal = preprocess_uploaded_image(frontal_image)
    lateral = preprocess_uploaded_image(lateral_image)

    memory = encoder_model([frontal, lateral], training=False)

    tokens = np.zeros((1, MAX_LENGTH - 1), dtype=np.int32)
    tokens[0, 0] = start_id

    for t in range(MAX_LENGTH - 2):
        logits = decoder_model(
            [tf.convert_to_tensor(tokens), memory],
            training=False
        ).numpy()

        step_logits = logits[:, t, :]
        step_logits[:, 0] = -1e9

        if 0 <= unk_id < step_logits.shape[-1]:
            step_logits[:, unk_id] = -1e9

        next_token = int(np.argmax(step_logits[0]))
        tokens[0, t + 1] = next_token

        if next_token == end_id:
            break

    words = []
    for token_id in tokens[0]:
        token_id = int(token_id)

        if token_id == end_id:
            break

        if token_id != 0 and token_id != start_id:
            word = tokenizer.index_word.get(token_id, "")
            if word:
                words.append(word)

    report = " ".join(words).strip()

    if not report:
        return "The model returned an empty report. Please check the model and tokenizer files."

    return report

# 7) Gradio interface
demo = gr.Interface(
    fn=generate_report,
    inputs=[
        gr.Image(type="pil", label="Frontal X-ray (optional)"),
        gr.Image(type="pil", label="Lateral X-ray (optional)")
    ],
    outputs=gr.Textbox(
        label="Generated Report",
        lines=14
    ),
    title="Chest X-ray Report Generator",
    description=(
        "Upload both X-rays for the intended two-view input. "
        "If you upload only one image, it will automatically be used "
        "for both model inputs."
    ),
    flagging_mode="never"
)

demo.launch(share=True, debug=True)

